In [1]:
import numpy as np
import pandas as pd

from scipy.stats import chi2

# Chi-square test

## Chi-square goodness-of-fit test

Historically, customers are expected to use payment methods in these proportions:
- Card: 40%
- PayPal: 30%
- Bank Transfer: 20%
- Cash: 10%
---
New sample of customers is collected                                                                                                                           
Reasearch Question:
- Does the current payment method distribution differ from the historical expected distribution ?
---
Indentifying the correct Chi-square test
- I use the `Chi-square goodness-of-fit test` because there is one categorical variable, and I want to compare its observed distribution with a specified expected distribution.
- I choose `Righ-tailed Chi-square goodness-of-fit test` because the test statistic squares every difference:
---
Hypothesis
- H₀ : The current payment-method distribution follows (0.40, 0.30, 0.20, 0.10)
- Hₐ : THe current payment-method distribution differs from (0.40, 0.30, 0.20, 0.10)
---
Significance level
- α = 0.05 we accpet `5%` Type I error
---
Formula:
- χ² = Σ ((Oᵢ - Eᵢ)² / Eᵢ)

In [2]:
customers_df = pd.read_csv('data/01_chi_square_goodness_of_fit.csv')
customers_df.head()

,customer_id,payment_method
0,1,PayPal
1,2,Bank Transfer
2,3,Bank Transfer
3,4,Card
4,5,PayPal


In [3]:
customers_df.shape

(240, 2)

In [4]:
sample_n = len(customers_df)
payment_type_alpha = 0.05
df_value = 4 - 1 

In [5]:
observed_counts = customers_df['payment_method'].value_counts()
observed_counts

payment_method
Card             120
PayPal            60
Bank Transfer     42
Cash              18
Name: count, dtype: int64

In [6]:
# expected proportions
expected_proportions = pd.Series(
    {
        'Card': 0.40,
        'PayPal': 0.30,
        'Bank Transfer': 0.20,
        'Cash': 0.10
    }
)

expected_counts = expected_proportions * sample_n
expected_counts

Card             96.0
PayPal           72.0
Bank Transfer    48.0
Cash             24.0
dtype: float64

In [7]:
# Calculatring Chi-square goodness-of-fit test by counts
payment_type_observed_chi_value = np.sum(
    ((observed_counts - expected_counts) ** 2) / expected_counts
)

print(payment_type_observed_chi_value)

10.25


In [8]:
# critical values
payment_type_critical_value = chi2.ppf(
    1 - payment_type_alpha,
    df=df_value
)
print("Critical Value: ", payment_type_critical_value)

# making a decision by critical value
if payment_type_observed_chi_value > payment_type_critical_value:
    print("Reject H₀")
else:
    print("Fail to reject H₀")

Critical Value:  7.814727903251179
Reject H₀


In [9]:
# p-value
payment_type_p_value = chi2.sf(payment_type_observed_chi_value, df=df_value)
print("P-value: ", payment_type_p_value)

if payment_type_p_value < payment_type_alpha:
    print("Reject H₀")
else:
    print("Fail to reject H₀")

P-value:  0.016556321391633846
Reject H₀


At the` 5% significance level`, there is sufficient statistical evidence to conclude that the current population payment-method distribution differs from the historical expected distribution.

## Chi-square test of independence

A company wants to know whether website conversion is associated with device type.                                                                                
Variables:
- device_type: Mobile or Desktop
- converted: 1 = Yes, 0 = No
---
Research question:
- Is conversion behavior associated with device type?
--- 
Indentifying the correct Chi-square test 
- I use the `Chi-square test of independence` because there are two categorical variables and I want to know whether they are associated.
---
Hypothesis:
- H₀ : Device type and conversion are independent
- Hₐ : Device type and conversion are associated
---
Significance level
- α = 0.05 we accpet 5% Type I error
---
Formula:
- Independent: `Eᵢⱼ = (Row total × Column total) / Grand total`
-  degrees of freem: `df_value = (r - 1) * (c - 1)`

In [10]:
devices_df = pd.read_csv("data/02_chi_square_independence.csv")
devices_df.head()

,visitor_id,device_type,converted
0,81,Mobile,0
1,164,Mobile,0
2,151,Mobile,0
3,245,Desktop,0
4,12,Mobile,1


In [11]:
devices_df.shape

(300, 3)

In [12]:
devices_df['device_type'].value_counts()

device_type
Mobile     180
Desktop    120
Name: count, dtype: int64

In [13]:
devices_df['converted'].value_counts()

converted
0    207
1     93
Name: count, dtype: int64

In [14]:
# Creating an observed categorical table
observed_df = pd.crosstab(
    devices_df['device_type'],
    devices_df['converted']
)

observed_df 

converted,0,1
device_type,,
Desktop,72,48
Mobile,135,45


In [15]:
#  calculate the expected count for every cell assuming H₀ is True by the formula: Eᵢⱼ = (Row total × Column total) / Grand total
row_totals = observed_df.sum(axis=1)
col_totals = observed_df.sum()
grand_total = observed_df.values.sum()

expected_table = np.outer(
    row_totals,
    col_totals
) / grand_total


expected_df = pd.DataFrame(
    expected_table,
    index=observed_df.index,
    columns=observed_df.columns
)

expected_df

converted,0,1
device_type,,
Desktop,82.8,37.2
Mobile,124.2,55.8


In [16]:
# calculating Chi-square goodness-of-fit test: χ² = Σ ((Oᵢ - Eᵢ)² / Eᵢ)
chi_square_cells = (
    (observed_df - expected_df) ** 2
) / expected_df

devices_chi_square_observed = chi_square_cells.sum().sum()

print("Chi-square contributions:")
print(chi_square_cells)
print("-----------------")
print("Observed Chi-square:", devices_chi_square_observed)

Chi-square contributions:
converted           0         1
device_type                    
Desktop      1.408696  3.135484
Mobile       0.939130  2.090323
-----------------
Observed Chi-square: 7.573632538569421


In [17]:
devices_alpha = 0.05
# degrees of freedom: df_value = (r - 1) * (c - 1)
devices_df_value = (observed_df.shape[0] - 1) * (observed_df.shape[1] - 1)
devices_df_value

1

In [18]:
# critical values 
devices_critical_value = chi2.ppf(
    1 - devices_alpha,
    df=devices_df_value
)
print(f"Critical Value: {devices_critical_value}")

# making decision by critical value
if devices_chi_square_observed > devices_critical_value:
    print("Reject H₀")
else:
    print("Fail to reject H₀")

Critical Value: 3.841458820694124
Reject H₀


In [19]:
# p-Value 
devices_p_value = chi2.sf(devices_chi_square_observed, df=devices_df_value)
print(f"Devices p_value: {devices_p_value}")

if devices_p_value < devices_alpha:
    print("Reject H₀")
else:
    print("Fail to reject H₀")


Devices p_value: 0.005922829700135666
Reject H₀


At the `5% significance level`, there is sufficient statistical evidence to conclude that conversion behavior is associated with device type.